# Reducing Unsupported and Fabricated Output
**Module 2: Reliable Prompting and Structured Outputs**

When we ask an LLM to extract fields from text, it sometimes:

- **fabricates** a value that is not in the text (an order ID, a date, a price), or
- makes an **unsupported** claim (a product brand the customer never mentioned).

In this notebook we:

1. See the problem with a naive prompt.
2. Write a small checker that flags values not found in the ticket.
3. Fix the problem step by step:
    - allow `null` for unknown fields,
    - ground the model in the ticket text only,
    - restrict values with a Pydantic schema (enums and patterns),
    - ask for evidence quotes and verify them in code,
    - add a few-shot example with missing information,
    - retry when validation fails, and flag what needs clarification.
4. Compare the naive and improved results.

> **Mock mode:** if no `OPENAI_API_KEY` is set, the notebook uses saved sample responses so every cell still runs. Set the key to see live model output.

## 1. Setup
Install the libraries once. `python-dotenv` lets you keep the API key in a `.env` file.

In [ ]:
# Run once (remove the # to install)
# !pip install openai pydantic python-dotenv

In [ ]:
import os, json
from typing import Literal, Optional
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads OPENAI_API_KEY from a .env file if present
except ImportError:
    pass

USE_MOCK = not os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4o-mini"
print("Mock mode" if USE_MOCK else f"Live mode with {MODEL}")

## 2. Sample support tickets
Three tickets with different amounts of information:

| Ticket | What it contains |
|---|---|
| T1 | Order ID, product, issue, urgency: everything is stated |
| T2 | Product type and issue, but **no order ID** |
| T3 | Very vague: almost nothing is stated |

In [ ]:
tickets = {
    "T1": "Order SK100001: my boAt Rockerz 450 headphones arrived with a cracked ear cup. Please replace asap.",
    "T2": "Hi, my headphones started crackling after 3 days. I want a refund.",
    "T3": "Where is my parcel??",
}
for tid, text in tickets.items():
    print(tid, "->", text)

## 3. A helper to call the model
`call_llm` sends a system prompt and a user message and returns the reply as text.

- `temperature=0` makes the output more consistent (it does **not** stop fabrication by itself).
- `response_format={"type": "json_object"}` asks the model to return valid JSON.
- In mock mode it returns a saved response for that step instead of calling the API.

In [ ]:
# Saved responses used only in mock mode, typical of what gpt-4o-mini returns for each step
MOCK = {
    ("naive", "T1"): {"order_id": "SK100001", "product": "boAt Rockerz 450 headphones", "issue": "damaged",
                      "purchase_date": "2026-11-10", "refund_amount": 1499, "urgency": "high"},
    ("naive", "T2"): {"order_id": "SK482913", "product": "boAt Rockerz 450", "issue": "defective",
                      "purchase_date": "2026-11-12", "refund_amount": 1499, "urgency": "medium"},
    ("naive", "T3"): {"order_id": "SK100245", "product": "parcel", "issue": "not_delivered",
                      "purchase_date": "2026-11-15", "refund_amount": None, "urgency": "high"},
    ("grounded", "T1"): {"order_id": "SK100001", "product": "boAt Rockerz 450 headphones", "issue": "damaged", "urgency": "high"},
    ("grounded", "T2"): {"order_id": None, "product": "headphones", "issue": "defective", "urgency": "medium"},
    ("grounded", "T3"): {"order_id": None, "product": None, "issue": "not_delivered", "urgency": "medium"},
    ("evidence", "T1"): {"intent": "replacement_request", "order_id": "SK100001", "product": "boAt Rockerz 450 headphones",
                         "issue": "damaged", "issue_evidence": "arrived with a cracked ear cup",
                         "urgency": "high", "urgency_evidence": "asap", "missing_fields": [], "needs_clarification": False},
    ("evidence", "T2"): {"intent": "refund_request", "order_id": None, "product": "headphones",
                         "issue": "defective", "issue_evidence": "sound is broken",          # invented quote
                         "urgency": "medium", "urgency_evidence": None, "missing_fields": ["order_id"], "needs_clarification": True},
    ("evidence_retry", "T2"): {"intent": "refund_request", "order_id": None, "product": "headphones",
                         "issue": "defective", "issue_evidence": "started crackling after 3 days",
                         "urgency": "medium", "urgency_evidence": None, "missing_fields": ["order_id"], "needs_clarification": True},
    ("evidence", "T3"): {"intent": "order_status", "order_id": None, "product": None,
                         "issue": "not_delivered", "issue_evidence": "Where is my parcel",
                         "urgency": "medium", "urgency_evidence": None, "missing_fields": ["order_id"], "needs_clarification": True},
}

def call_llm(system_prompt, user_message, step, ticket_id):
    if USE_MOCK:
        return json.dumps(MOCK[(step, ticket_id)])
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": system_prompt},
                  {"role": "user", "content": user_message}],
    )
    return response.choices[0].message.content

## 4. The problem: a naive prompt
The prompt below asks for fields but never says where the values may come from, or what to do when a value is missing.

In [ ]:
NAIVE_PROMPT = """Extract the order details from the customer ticket as JSON with these keys:
order_id, product, issue, purchase_date, refund_amount, urgency."""

naive_results = {}
for tid, text in tickets.items():
    naive_results[tid] = json.loads(call_llm(NAIVE_PROMPT, text, "naive", tid))
    print(tid, json.dumps(naive_results[tid], indent=2), "\n")

Look at **T2** and **T3**. Neither ticket contains an order ID, a purchase date or a refund amount, yet the output has all three. These are **fabricated** values: they look real, which is exactly why they are dangerous.

## 5. A simple checker for unsupported values
Before fixing anything, we need a way to **measure** the problem.

`find_unsupported` looks at every text or number value and checks whether it appears in the ticket. It ignores:
- `None` (an honest "not stated"),
- fields we expect the model to classify rather than copy, such as `issue` and `urgency`.

This is deliberately simple; it is enough to show the difference.

In [ ]:
CLASSIFIED_FIELDS = {"issue", "urgency", "intent", "needs_clarification", "missing_fields"}

def find_unsupported(result, ticket_text):
    """Return the fields whose value does not appear in the ticket text."""
    unsupported = []
    for field, value in result.items():
        if value is None or field in CLASSIFIED_FIELDS or field.endswith("_evidence"):
            continue
        if str(value).lower() not in ticket_text.lower():
            unsupported.append(f"{field}={value}")
    return unsupported

for tid, result in naive_results.items():
    print(tid, "unsupported:", find_unsupported(result, tickets[tid]))

## 6. Fix 1 and 2: allow null, and ground the model in the ticket
Two small prompt changes make a big difference:

1. **Give the model a legal way to say "unknown"**: return `null` when a field is not stated.
2. **Ground it**: put the ticket inside `<ticket>` tags and say *use only this text*.

We also drop `purchase_date` and `refund_amount`. A ticket is the wrong source for these values; they belong to the order system, which we will query with a tool in Module 3.

In [ ]:
GROUNDED_PROMPT = """You extract fields from a customer support ticket.

Rules:
- Use ONLY the text inside <ticket></ticket>. Do not use outside knowledge about brands, prices or dates.
- If a field is not explicitly stated in the ticket, return null. Never guess.
- Return JSON with keys: order_id, product, issue, urgency.
- issue must be one of: defective, damaged, wrong_item, not_delivered, other.
- urgency must be one of: low, medium, high."""

grounded_results = {}
for tid, text in tickets.items():
    user_message = f"<ticket>{text}</ticket>"
    grounded_results[tid] = json.loads(call_llm(GROUNDED_PROMPT, user_message, "grounded", tid))
    print(tid, grounded_results[tid])
    print("   unsupported:", find_unsupported(grounded_results[tid], text))

Now `order_id` is `null` for T2 and T3, and "headphones" is no longer upgraded to a brand name. The checker finds nothing unsupported.

## 7. Fix 3: a Pydantic schema with enums and patterns
Prompt rules are requests; **code** is enforcement. A Pydantic model checks every response:

- `Literal[...]` limits `issue`, `urgency` and `intent` to fixed values.
- `pattern` makes sure an order ID looks like `SK` followed by 6 digits.
- `Optional[...]` means a field may be `None` (unknown).

In [ ]:
class TicketExtraction(BaseModel):
    intent: Literal["order_status", "refund_request", "replacement_request", "complaint", "other"]
    order_id: Optional[str] = Field(default=None, pattern=r"^SK[0-9]{6}$")
    product: Optional[str] = None
    issue: Literal["defective", "damaged", "wrong_item", "not_delivered", "other"]
    urgency: Literal["low", "medium", "high"]

# A quick test: an invented, badly formatted order ID is rejected
try:
    TicketExtraction(intent="refund_request", order_id="ORDER-12", issue="defective", urgency="high")
except ValidationError as e:
    print("Rejected:", e.errors()[0]["msg"])

# A made-up category is rejected too
try:
    TicketExtraction(intent="refund_request", issue="faulty audio hardware", urgency="high")
except ValidationError as e:
    print("Rejected:", e.errors()[0]["msg"][:80])

## 8. Fix 4: ask for evidence, then verify it in code
We ask the model to copy the exact words that support `issue` and `urgency`.
Then a validator checks that those words **really appear** in the ticket.
If the model invents a quote, validation fails.

The model also returns:
- `missing_fields`: what it could not find, and
- `needs_clarification`: whether we must ask the customer before acting.

In [ ]:
class GroundedExtraction(TicketExtraction):
    issue_evidence: Optional[str] = None      # exact words from the ticket
    urgency_evidence: Optional[str] = None
    missing_fields: list[str] = []
    needs_clarification: bool = False
    source_text: str                          # the ticket; added by OUR code, not by the model

    @model_validator(mode="after")
    def evidence_must_be_in_ticket(self):
        for name in ["issue_evidence", "urgency_evidence"]:
            quote = getattr(self, name)
            if quote and quote.lower() not in self.source_text.lower():
                raise ValueError(f"{name} '{quote}' is not in the ticket, so it is unsupported")
        return self

    @model_validator(mode="after")
    def order_id_must_be_in_ticket(self):
        if self.order_id and self.order_id not in self.source_text:
            raise ValueError(f"order_id {self.order_id} is not in the ticket, so it is fabricated")
        return self

## 9. Fix 5: a few-shot example with missing information
If every example shows all fields filled in, the model learns that fields are *always* filled.
So we include one example where most fields are `null`.

In [ ]:
EVIDENCE_PROMPT = """You extract fields from a customer support ticket.

Rules:
- Use ONLY the text inside <ticket></ticket>. Never guess or use outside knowledge.
- If a field is not explicitly stated, return null and list it in missing_fields.
- issue_evidence and urgency_evidence must be EXACT words copied from the ticket, or null.
- Set needs_clarification to true if order_id is missing.
- intent: order_status | refund_request | replacement_request | complaint | other
- issue: defective | damaged | wrong_item | not_delivered | other
- urgency: low | medium | high (use medium if not stated)

Return JSON with keys: intent, order_id, product, issue, issue_evidence,
urgency, urgency_evidence, missing_fields, needs_clarification.

Example 1
<ticket>Order SK100777 - the mixer jar is broken. Need it fixed today!</ticket>
{"intent":"replacement_request","order_id":"SK100777","product":"mixer","issue":"damaged",
 "issue_evidence":"the mixer jar is broken","urgency":"high","urgency_evidence":"Need it fixed today",
 "missing_fields":[],"needs_clarification":false}

Example 2 (information missing)
<ticket>still waiting...</ticket>
{"intent":"order_status","order_id":null,"product":null,"issue":"not_delivered",
 "issue_evidence":"still waiting","urgency":"medium","urgency_evidence":null,
 "missing_fields":["order_id","product"],"needs_clarification":true}"""

## 10. Fix 6: validate, and retry with the error message
`extract_with_retry`:

1. calls the model,
2. adds the ticket text and validates with `GroundedExtraction`,
3. on failure, sends the **exact validation error** back to the model and asks it to fix the output,
4. gives up after `max_retries` and returns `None`, so a human can review it instead of using a bad value.

In [ ]:
def extract_with_retry(ticket_id, text, max_retries=2):
    user_message = f"<ticket>{text}</ticket>"
    step = "evidence"
    for attempt in range(max_retries + 1):
        raw = call_llm(EVIDENCE_PROMPT, user_message, step, ticket_id)
        try:
            data = json.loads(raw)
            result = GroundedExtraction(**data, source_text=text)
            print(f"  attempt {attempt + 1}: valid")
            return result
        except (json.JSONDecodeError, ValidationError) as error:
            short_error = error.errors()[0]["msg"] if isinstance(error, ValidationError) else str(error)
            print(f"  attempt {attempt + 1}: failed -> {short_error}")
            user_message = (f"<ticket>{text}</ticket>\n\nYour previous answer was:\n{raw}\n"
                            f"It failed validation: {short_error}\n"
                            "Fix it. Copy evidence exactly from the ticket or use null.")
            step = "evidence_retry"
    return None    # send to human review

final_results = {}
for tid, text in tickets.items():
    print(tid)
    final_results[tid] = extract_with_retry(tid, text)

In mock mode, T2's first answer quotes *"sound is broken"*, words the customer never wrote. The validator catches it, the error goes back to the model, and the second attempt quotes the real text *"started crackling after 3 days"*.

In [ ]:
for tid, result in final_results.items():
    if result is None:
        print(tid, "-> sent to human review")
        continue
    shown = result.model_dump(exclude={"source_text"})
    print(tid, json.dumps(shown, indent=2))
    if result.needs_clarification:
        print(f"   Next step: ask the customer for {', '.join(result.missing_fields)}\n")

## 11. Before and after
We count unsupported values for the naive prompt and the final version.

In [ ]:
for tid, text in tickets.items():
    before = find_unsupported(naive_results[tid], text)
    final = final_results[tid]
    after = find_unsupported(final.model_dump(exclude={"source_text"}), text) if final else ["(human review)"]
    print(f"{tid}  naive: {len(before)} unsupported {before}")
    print(f"    final: {len(after)} unsupported {after}\n")

## 12. Summary

| Technique | What it stops | Where it lives |
|---|---|---|
| Allow `null` + `missing_fields` | Guessing to fill a field | Prompt + schema |
| Ground in `<ticket>` text only | Outside knowledge creeping in (brands, prices) | Prompt |
| Enums and patterns | Invented categories and badly formatted IDs | Pydantic schema |
| Evidence quotes, checked in code | Claims with no support in the text | Prompt + validator |
| Few-shot with missing info | The habit of filling every field | Prompt |
| Retry with the validation error | Accepting a bad first answer | Code |
| `needs_clarification` / human review | Acting on incomplete information | Code + workflow |

**Key idea:** the prompt *asks* for honest output; the schema and validators *enforce* it.

## 13. Exercise
1. Add a fourth ticket: `"I was charged twice for order SK100009, the power bank. Fix this now."` Run it through the naive prompt and the final version.
2. Add a new field `quantity` (an integer). Make sure it is `null` when the customer does not state a number.
3. Change `find_unsupported` so it also reports when an evidence quote is not found in the ticket.
4. **Discussion:** which fields should never come from the ticket at all, and which system should supply them instead? (Hint: Module 3, Tools.)